In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1us5ETVzI6ddh44sjTyglmTY0r9ogcmsd", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_01_intro.mp3"))

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Multi-Agent Coordinator Patterns — Hands-On Lab

In this notebook, we build a **multi-agent coordinator system** — the hub-and-spoke architecture where one coordinator delegates to specialized subagents with isolated context.

By the end of this notebook, you will be able to:
- Implement a coordinator that decomposes tasks and delegates to subagents
- Demonstrate why subagents must have **isolated context**
- Build the Task tool pattern for spawning subagents
- Identify over-decomposition and fix it

## AI Teaching Assistant

**[Open AI Teaching Assistant](https://pods.vizuara.ai/courses/claude-certified-architect/agentic-architecture/practice/2/assistant)**

In [ ]:
#@title 🎧 Listen: Why Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_why_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Imagine a hospital ER. A triage nurse assesses patients and routes them to specialists. The nurse does not treat anyone directly — she coordinates.

This is hub-and-spoke multi-agent architecture. One coordinator at the hub, specialist subagents as spokes. The exam tests this heavily, especially **isolated context**.

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition

In [ ]:
import json
import uuid
import sys
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

all_tools = ["lookup_order", "check_shipping", "process_refund", "search_kb", "lookup_policy",
             "calculate_refund", "check_inventory", "update_ticket", "send_email"]

print(f"One agent with {len(all_tools)} tools:")
print("  - Context window wasted on irrelevant tool descriptions")
print("  - Hard to debug")
print()
print("Hub-and-spoke:")
print("  - Coordinator delegates to focused specialists")
print("  - Each specialist has 2-3 relevant tools")

In [ ]:
#@title 🎧 Listen: Agent Definitions
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_agent_definitions.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. Agent Definitions

In [ ]:
@dataclass
class AgentDefinition:
    name: str
    system_prompt: str
    tools: List[str]
    def __repr__(self): return f"Agent({self.name}, tools={self.tools})"

research_agent = AgentDefinition("Research Specialist", "Search and summarize information.", ["search_kb", "lookup_policy"])
analysis_agent = AgentDefinition("Policy Analyst", "Determine if actions are warranted.", ["calculate_refund", "check_eligibility"])
action_agent = AgentDefinition("Action Executor", "Execute approved actions.", ["process_refund", "update_ticket", "send_email"])

for a in [research_agent, analysis_agent, action_agent]:
    print(f"  {a}")

In [ ]:
#@title 🎧 Listen: Isolated Context
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_isolated_context.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Isolated Context — The Critical Principle

Subagents do NOT inherit coordinator history. Each starts fresh.

In [ ]:
class SubagentRunner:
    def __init__(self, agent_def):
        self.agent_def = agent_def
        self.log = []

    def run(self, prompt, context=None):
        self.log.append({"agent": self.agent_def.name, "prompt_len": len(prompt),
                        "context_keys": list(context.keys()) if context else [], "inherited": False})
        return self._sim(prompt)

    def _sim(self, prompt):
        if "policy" in prompt.lower():
            return json.dumps({"policy": "30-day return window", "refund_type": "Full refund"})
        elif "eligible" in prompt.lower():
            return json.dumps({"eligible": True, "amount": 79.99})
        elif "process" in prompt.lower():
            return json.dumps({"success": True, "refund_id": f"REF-{uuid.uuid4().hex[:6]}"})
        return json.dumps({"result": "Done"})

runner = SubagentRunner(research_agent)
coordinator_history = [f"Customer {i} msg..." for i in range(20)]

print(f"Coordinator has {len(coordinator_history)} messages")
result = runner.run("Research return policy for Electronics", {"category": "Electronics"})
print(f"Subagent got: ONLY the prompt (no coordinator history)")
print(f"Log: {runner.log[-1]}")

### Why It Matters

In [ ]:
coord_size = sum(len(m) for m in coordinator_history)
sub_size = len("Research return policy for Electronics")

print(f"Coordinator context: ~{coord_size} chars")
print(f"Subagent prompt: ~{sub_size} chars")
print(f"Overhead ratio: {coord_size // max(sub_size, 1)}x if not isolated")
print()
print("Problems without isolation:")
print("  1. Wasted context window")
print("  2. Confusion from irrelevant data")
print("  3. Data leakage between customers")
print()
print("Exam: subagents ALWAYS get isolated context.")

In [ ]:
#@title 🎧 Listen: Coordinator
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_coordinator.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. The Coordinator

In [ ]:
class Coordinator:
    def __init__(self):
        self.agents = {"research": SubagentRunner(research_agent),
                      "analysis": SubagentRunner(analysis_agent),
                      "action": SubagentRunner(action_agent)}

    def handle(self, request):
        subtasks = [
            {"id": "research", "agent": "research",
             "desc": f"Research policy for {request.get('category')}",
             "prompt": f"Research return policy for {request.get('category')}"},
            {"id": "analysis", "agent": "analysis",
             "desc": f"Check eligibility for {request.get('order_id')}",
             "prompt": f"Is refund eligible for {request.get('order_id')}? Complaint: {request.get('complaint')}"},
            {"id": "action", "agent": "action",
             "desc": "Process refund",
             "prompt": f"Process refund for {request.get('order_id')}"},
        ]

        print(f"Decomposed into {len(subtasks)} subtasks:")
        results = {}
        for st in subtasks:
            result = self.agents[st["agent"]].run(st["prompt"])
            results[st["id"]] = json.loads(result)
            print(f"  [{st['agent']}] {st['desc']}: done")

        return {"order": request.get("order_id"),
                "policy": results.get("research", {}).get("policy"),
                "eligible": results.get("analysis", {}).get("eligible"),
                "refund": results.get("action", {})}

coord = Coordinator()
result = coord.handle({"customer_id": "C-789", "order_id": "ORD-1234",
                       "category": "Electronics", "complaint": "Item broke after 2 weeks"})
print(f"\nResult: {json.dumps(result, indent=2)}")

In [ ]:
#@title 🎧 Listen: Over Decomposition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_over_decomposition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Over-Decomposition

In [ ]:
over = ["Email Validator", "Phone Formatter", "Zip Validator", "Name Normalizer", "Address Checker"]
proper = ["Data Validator (all 5 tools)"]

print(f"Over-decomposed: {len(over)} agents = {len(over)*500}ms")
print(f"Properly decomposed: {len(proper)} agent = {len(proper)*500}ms")
print(f"Savings: {(len(over)-len(proper))*500}ms")
print()
print("Rule: Decompose into MEANINGFUL subtasks, not tiny units.")

In [ ]:
#@title 🎧 Listen: Todo Exercises
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_todo_exercises.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. TODO Exercises

### Exercise 1: Add Communication Agent

In [ ]:
# ============ TODO ============
# Add agent with tools: ["send_email", "send_sms", "update_ticket"]
# Add notification step to coordinator after refund.
# YOUR CODE HERE
# ==============================

### Exercise 2: Context Isolation Bug Demo

In [ ]:
# ============ TODO ============
# Show what goes wrong when full coordinator history is passed to subagent.
# YOUR CODE HERE
# ==============================

### Exercise 3: Parallel Execution

In [ ]:
# ============ TODO ============
# Use ThreadPoolExecutor for independent subtasks.
# YOUR CODE HERE
# ==============================

In [ ]:
#@title 🎧 Listen: Task Tool
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_task_tool.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. Task Tool Pattern

In [ ]:
class TaskTool:
    def __init__(self): self.spawned = []
    def invoke(self, agent_def, prompt):
        runner = SubagentRunner(agent_def)
        result = runner.run(prompt)
        self.spawned.append({"agent": agent_def.name, "prompt": prompt[:60]})
        return result

tt = TaskTool()
r = tt.invoke(research_agent, "Research return policy for Electronics.")
print(f"Spawned: {tt.spawned[-1]}")
print(f"Result: {r}")
print("Key: Subagent got ONLY the prompt. No coordinator history.")

In [ ]:
#@title 🎧 Listen: Exam Practice
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_exam_practice.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Exam Practice

In [ ]:
for i, (q, a, w) in enumerate([
    ("Coordinator has 20 customers, spawns subagent for #21. What context?",
     "Only the specific prompt", "Isolated context, no history."),
    ("5 single-field validators. Problem?",
     "Over-decomposition", "One validator with all 5 tools is better."),
    ("Coordinator's 3 responsibilities?",
     "Decompose, Delegate, Aggregate", "The three phases."),
    ("Subagent needs order history. How?",
     "Coordinator passes it in the prompt", "Explicit context passing."),
], 1):
    print(f"Q{i}: {q}\n   {a} — {w}\n")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 10. Key Takeaways

**Task Statements 1.2, 1.3:**
- Hub-and-spoke: coordinator + specialist subagents
- **Subagents have isolated context**
- Coordinator: decompose, delegate, aggregate
- Task tool spawns with explicit context
- Avoid over-decomposition

In [ ]:
print("Notebook complete! Next: Hooks & Enforcement (NB 3)")